In [81]:
import numpy as np


def cleanup():
    import gc
    import torch
    torch.mps.empty_cache()
    gc.collect()

In [59]:
cleanup()

In [60]:
import torch
import torch.nn as nn

class VGG16(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG16, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),


            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),


            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),


            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),


            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = VGG16(num_classes=10).to(device)
print(f"Model is running on: {device}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

Model is running on: mps
Total Parameters: 134,301,514


In [61]:
loss_fn = nn.CrossEntropyLoss()

In [62]:
optimiser = torch.optim.Adam(model.parameters(),lr=0.0001)

In [63]:
from torchvision import transforms
import torchvision

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [64]:
shared_root='/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/1.LeNet-5/data/'

train_set = torchvision.datasets.CIFAR10(root=shared_root, train=True, download=True, transform=transform)

In [65]:
print(next(iter(train_set)))

(tensor([[[-0.5373, -0.5373, -0.5373,  ...,  0.1608,  0.1608,  0.1608],
         [-0.5373, -0.5373, -0.5373,  ...,  0.1608,  0.1608,  0.1608],
         [-0.5373, -0.5373, -0.5373,  ...,  0.1608,  0.1608,  0.1608],
         ...,
         [ 0.3882,  0.3882,  0.3882,  ..., -0.0353, -0.0353, -0.0353],
         [ 0.3882,  0.3882,  0.3882,  ..., -0.0353, -0.0353, -0.0353],
         [ 0.3882,  0.3882,  0.3882,  ..., -0.0353, -0.0353, -0.0353]],

        [[-0.5137, -0.5137, -0.5137,  ..., -0.0275, -0.0275, -0.0275],
         [-0.5137, -0.5137, -0.5137,  ..., -0.0275, -0.0275, -0.0275],
         [-0.5137, -0.5137, -0.5137,  ..., -0.0275, -0.0275, -0.0275],
         ...,
         [ 0.1294,  0.1294,  0.1294,  ..., -0.2784, -0.2784, -0.2784],
         [ 0.1294,  0.1294,  0.1294,  ..., -0.2784, -0.2784, -0.2784],
         [ 0.1294,  0.1294,  0.1294,  ..., -0.2784, -0.2784, -0.2784]],

        [[-0.5059, -0.5059, -0.5059,  ..., -0.1922, -0.1922, -0.1922],
         [-0.5059, -0.5059, -0.5059,  ..., -

In [68]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set,shuffle= True, batch_size=30)

In [67]:
from ptflops import get_model_complexity_info

test_model_parameter = VGG16(num_classes=10)

macs, params = get_model_complexity_info(test_model_parameter, (3, 244, 244), as_strings=True, print_per_layer_stat=True)

VGG16(
  134.3 M, 100.000% Params, 18.14 GMac, 99.872% MACs, 
  (features): Sequential(
    14.71 M, 10.956% Params, 18.02 GMac, 99.213% MACs, 
    (0): Conv2d(1.79 k, 0.001% Params, 106.69 MMac, 0.588% MACs, 3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(0, 0.000% Params, 3.81 MMac, 0.021% MACs, inplace=True)
    (2): Conv2d(36.93 k, 0.027% Params, 2.2 GMac, 12.107% MACs, 64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(0, 0.000% Params, 3.81 MMac, 0.021% MACs, inplace=True)
    (4): MaxPool2d(0, 0.000% Params, 3.81 MMac, 0.021% MACs, kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(73.86 k, 0.055% Params, 1.1 GMac, 6.053% MACs, 64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(0, 0.000% Params, 1.91 MMac, 0.010% MACs, inplace=True)
    (7): Conv2d(147.58 k, 0.110% Params, 2.2 GMac, 12.096% MACs, 128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(0, 0.000% P

In [69]:
"""from tqdm.notebook import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_Metrics = SummaryWriter()

latency_per_image = []
global_i = 0

pbar = tqdm(train_loader)

total=len(train_loader)

all_preds = []
all_labels = []

for image,label in pbar:

    image, label = image.to(device), label.to(device)

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    _, predicted = torch.max(y_pred.data, 1)
    correct = (predicted == label).sum().item()
    accuracy = correct / label.size(0)

    end_time = time.time()

    latency_per_image.append(end_time - start_time)

    LeNet5_Metrics.add_scalar("Loss/train - VGG16 IT599 Ben", loss.item(), global_i)
    LeNet5_Metrics.add_scalar("Accuracy/train - VGG16 IT599 Ben", accuracy, global_i)

    all_preds.extend(predicted.cpu())
    all_labels.extend(label.cpu())

    global_i += 1

    print(global_i/total)


LeNet5_Metrics.close()

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")"""

  0%|          | 0/1667 [00:00<?, ?it/s]

0.0005998800239952009
0.0011997600479904018
0.001799640071985603
0.0023995200959808036
0.002999400119976005
0.003599280143971206
0.004199160167966407
0.004799040191961607
0.005398920215956809
0.00599880023995201
0.0065986802639472104
0.007198560287942412
0.007798440311937612
0.008398320335932814
0.008998200359928014
0.009598080383923215
0.010197960407918417
0.010797840431913617
0.011397720455908818
0.01199760047990402
0.01259748050389922
0.013197360527894421
0.013797240551889621
0.014397120575884824
0.014997000599880024
0.015596880623875225
0.016196760647870425
0.016796640671865627
0.01739652069586083
0.017996400719856028
0.01859628074385123
0.01919616076784643
0.01979604079184163
0.020395920815836834
0.020995800839832032
0.021595680863827234
0.022195560887822437
0.022795440911817635
0.023395320935812838
0.02399520095980804
0.02459508098380324
0.02519496100779844
0.02579484103179364
0.026394721055788842
0.026994601079784044
0.027594481103779243
0.028194361127774445
0.028794241151769647

KeyboardInterrupt: 

In [72]:
#torch.save(model.state_dict(), "/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/3.VGG16/model_VGG_50%")

In [74]:

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")

Latency: 902.58 ms | Throughput: 1.11 items/sec


In [ ]:
"""if epoch % 1 == 0:
    torch.save(model.state_dict(), f"vgg16_epoch{epoch}.pth")"""

In [ ]:
#python3 -m tensorboard.main --logdir="/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/3.VGG16/runs"

In [85]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.32      0.26      0.29      2336
           1       0.34      0.25      0.29      2328
           2       0.21      0.05      0.08      2289
           3       0.21      0.14      0.17      2366
           4       0.22      0.10      0.14      2274
           5       0.15      0.23      0.18      2334
           6       0.21      0.26      0.23      2289
           7       0.15      0.37      0.21      2344
           8       0.28      0.26      0.27      2308
           9       0.31      0.26      0.28      2352

    accuracy                           0.22     23220
   macro avg       0.24      0.22      0.21     23220
weighted avg       0.24      0.22      0.22     23220



In [87]:
from tqdm.notebook import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_Metrics = SummaryWriter()

latency_per_image = []
global_i = 0

pbar = tqdm(train_loader)

total=len(train_loader)

all_preds = []
all_labels = []

j = 0

for image,label in pbar:

    j = j + 1

    if j < 773:
        continue

    image, label = image.to(device), label.to(device)

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    _, predicted = torch.max(y_pred.data, 1)
    correct = (predicted == label).sum().item()
    accuracy = correct / label.size(0)

    end_time = time.time()

    latency_per_image.append(end_time - start_time)

    LeNet5_Metrics.add_scalar("Loss/train - VGG16 IT599 Ben", loss.item(), global_i)
    LeNet5_Metrics.add_scalar("Accuracy/train - VGG16 IT599 Ben", accuracy, global_i)

    all_preds.extend(predicted.cpu())
    all_labels.extend(label.cpu())

    global_i += 1

    print(global_i/total)


LeNet5_Metrics.close()

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")

  0%|          | 0/1667 [00:00<?, ?it/s]

0.0005998800239952009
0.0011997600479904018
0.001799640071985603
0.0023995200959808036
0.002999400119976005
0.003599280143971206
0.004199160167966407
0.004799040191961607
0.005398920215956809
0.00599880023995201
0.0065986802639472104
0.007198560287942412
0.007798440311937612
0.008398320335932814
0.008998200359928014
0.009598080383923215
0.010197960407918417
0.010797840431913617
0.011397720455908818
0.01199760047990402
0.01259748050389922
0.013197360527894421
0.013797240551889621
0.014397120575884824
0.014997000599880024
0.015596880623875225
0.016196760647870425
0.016796640671865627
0.01739652069586083
0.017996400719856028
0.01859628074385123
0.01919616076784643
0.01979604079184163
0.020395920815836834
0.020995800839832032
0.021595680863827234
0.022195560887822437
0.022795440911817635
0.023395320935812838
0.02399520095980804
0.02459508098380324
0.02519496100779844
0.02579484103179364
0.026394721055788842
0.026994601079784044
0.027594481103779243
0.028194361127774445
0.028794241151769647

In [88]:
torch.save(model.state_dict(), "/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/3.VGG16/model_VGG_100%")

In [89]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.53      0.50      0.51      2680
           1       0.53      0.60      0.56      2697
           2       0.31      0.20      0.24      2658
           3       0.30      0.27      0.28      2624
           4       0.35      0.31      0.33      2704
           5       0.37      0.37      0.37      2663
           6       0.40      0.50      0.44      2641
           7       0.46      0.52      0.48      2746
           8       0.53      0.54      0.53      2736
           9       0.47      0.51      0.49      2691

    accuracy                           0.43     26840
   macro avg       0.42      0.43      0.42     26840
weighted avg       0.42      0.43      0.43     26840

